# 🌀 One polymer, three experiments — Carreau fitting meets Cox–Merz and Delaware–Rutgers

*Steady shear says how a polymer solution flows; oscillation says how it relaxes. For a
high-molecular-weight linear polymer in water, the two stories should be the same story —
and with `rheofit` you can check, quantitatively, that they are.*

*Reproduces the [rheofit case study](https://rheofit.readthedocs.io/en/latest/walkthrough-carreau.html),
run live in JupyterLite.*

**Run each cell in order** (▶). The first cell installs `rheofit` in your browser — no server involved.


In [ ]:
%pip install -q rheofit "rheopy-rheodata>=0.1.1"


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rheofit
import rheodata
print("rheofit", rheofit.__version__ if hasattr(rheofit, "__version__") else "installed")
print("rheodata", getattr(rheodata, "__version__", "installed"))


## 🧪 The dataset

Equilibrium experiments on a **high-molecular-weight linear polymer
in aqueous solution** at **25 °C** — three experiments, all from the
[rheodata](https://github.com/rheopy/rheodata) library (v0.1.1+): the flow curve
(`caggioni_linear_polymer_flow`), the amplitude sweep
(`caggioni_linear_polymer_amplitude_sweep`) and the frequency sweep
(`caggioni_linear_polymer_frequency_sweep`). No file downloads needed.


In [ ]:
# all three experiments from rheodata
flow = rheodata.to_rheofit("caggioni_linear_polymer_flow", "linear_polymer")
amp  = rheodata.load("caggioni_linear_polymer_amplitude_sweep").df  # 41-pt strain sweep @ 1 rad/s
freq = rheodata.load("caggioni_linear_polymer_frequency_sweep").df  # 31-pt frequency sweep
print(f"flow curve: {len(flow)} points | amplitude sweep: {len(amp)} points | frequency sweep: {len(freq)} points")


## 📐 The Carreau model

The Carreau model describes a fluid with a **zero-shear plateau** that thins toward a
power law at high shear rates:

η(γ̇) = η₀ [1 + (λγ̇)²]^((n−1)/2)

Each parameter has a direct physical reading: **η₀** is the zero-shear viscosity of the
fully entangled network, **λ** is the characteristic relaxation time (thinning sets in at
γ̇_c = 1/λ), and **n** is the high-shear power-law exponent. Fitting the flow curve is
one line:


In [ ]:
x = flow["Shear rate / 1/s"].to_numpy()
y = flow["Stress / Pa"].to_numpy()
eta = y / x  # viscosity from stress/shear-rate
o = np.argsort(x); xa, etaa = x[o], eta[o]  # ascending, for interpolation

res = rheofit.fit(flow, "carreau", effort="thorough", seed=0)
p = {k: v["value"] for k, v in res["params"].items()}
print("RedChi2 = %.3e, cond = %.1f" % (res["redchi"], res["cond"]))
for k, v in res["params"].items():
    print("%-10s = %10.4g ± %.1f%%" % (k, v["value"], 100 * v["stderr"] / v["value"]))

g = np.logspace(-2, 3, 300)  # shear-rate grid for smooth curves
eta_fit = p["eta_0"] * (1 + (p["lambda_val"] * g) ** 2) ** ((p["n"] - 1) / 2)
s_fit = eta_fit * g


**Expect:** η₀ ≈ 1.99 Pa·s (±1.3 %), λ ≈ 0.199 s (±6.2 %), n ≈ 0.414 (±2.3 %),
RedChi² ≈ 3.85×10⁻³ — a clean, fully identified fit (condition number ≈ 9.5).


In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(7, 6), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})
a1.loglog(x, y, "ko", ms=4, label="data (flow curve)")
a1.loglog(g, s_fit, "b-", lw=2, label="Carreau fit")
a1.set_ylabel("stress / Pa"); a1.legend(); a1.grid(True, which="both", alpha=0.3)
a1.set_title("Carreau fit to the flow curve")
resid = (y - p["eta_0"] * x * (1 + (p["lambda_val"] * x) ** 2) ** ((p["n"] - 1) / 2)) / y * 100
a2.semilogx(x, resid, "bo", ms=4)
a2.axhline(0, color="k", lw=0.8)
a2.set_xlabel("shear rate / 1/s"); a2.set_ylabel("rel. resid. / %")
a2.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()


The residuals show a mild S-shape (±10 %) — the honest signature of a real polymer
solution against the idealized Carreau form, not a fitting failure.

## 🔁 Cox–Merz: the frequency sweep predicts the flow curve

In 1958, Cox and Merz noticed something remarkable for polymer melts and solutions: the
**steady-shear viscosity equals the magnitude of the complex viscosity** when shear rate
and angular frequency are matched —

η(γ̇) = |η*(ω)| at ω = γ̇

— even though one is a nonlinear steady measurement and the other comes from small,
linear oscillations. No deep theory demanded it; the data did. With |η*| = √(G′²+G″²)/ω
from the frequency sweep:


In [ ]:
w = freq["omega_rad/s"].to_numpy()
Gp = freq["Gp_Pa"].to_numpy()
Gpp = freq["Gpp_Pa"].to_numpy()
eta_star = np.sqrt(Gp ** 2 + Gpp ** 2) / w   # |eta*(omega)|

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(x, eta, "ko", ms=4, label="η(γ̇) — flow sweep")
ax.loglog(w, eta_star, "rs", ms=4, mfc="none", mew=1.5, label="|η*(ω)| — frequency sweep")
ax.loglog(g, eta_fit, "b-", lw=1.5, label="Carreau fit (flow curve)")
ax.set_xlabel("shear rate / 1/s = angular frequency / rad/s")
ax.set_ylabel("viscosity / Pa·s")
ax.set_title("Cox–Merz: |η*(ω)| overlaid on η(γ̇)")
ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

m = (w >= 0.1) & (w <= 100)
eta_interp = np.exp(np.interp(np.log(w[m]), np.log(xa), np.log(etaa)))
logrms = 100 * np.sqrt(np.mean((np.log(eta_interp) - np.log(eta_star[m])) ** 2))
print("Cox–Merz agreement: %.1f%% log-RMS over 0.1–100 s⁻¹" % logrms)


The two curves coincide to within **≈2 % (log-RMS)** over three decades, 0.1–100 s⁻¹.
For this linear polymer solution, Cox–Merz holds essentially exactly — the small-amplitude
oscillatory measurement predicts the nonlinear flow curve with no adjustable parameters.

## 📊 Delaware–Rutgers: the amplitude sweep predicts the flow curve too

Doraiswamy and co-workers generalized Cox–Merz to *other* shear deformations: in a strain
sweep at fixed frequency, the **dynamic viscosity plotted against the strain-rate amplitude**
recovers the steady-shear viscosity —

η(γ̇) = η′(γ₀ω) at γ̇ = γ₀ω

— the **Delaware–Rutgers rule**. The amplitude sweep ran at ω = 1 rad/s, so γ₀ω spans
0.001–10 s⁻¹ — right across the Carreau plateau and into the thinning regime:


In [ ]:
gamma0 = amp["strain_pct"].to_numpy() / 100   # strain amplitude (fraction)
omega_amp = 1.0                               # amplitude sweep ran at fixed 1 rad/s
g0w = gamma0 * omega_amp                       # strain-rate amplitude
eta_p = amp["Gpp_Pa"].to_numpy() / omega_amp  # dynamic viscosity eta' = G''/w

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(x, eta, "ko", ms=4, label="η(γ̇) — flow sweep")
ax.loglog(g0w, eta_p, "g^", ms=4, mfc="none", mew=1.5, label="η′(γ₀ω) — amplitude sweep")
ax.loglog(g, eta_fit, "b-", lw=1.5, label="Carreau fit (flow curve)")
ax.set_xlabel("shear rate / 1/s = strain-rate amplitude γ₀ω / 1/s")
ax.set_ylabel("viscosity / Pa·s")
ax.set_title("Delaware–Rutgers: η′(γ₀ω) overlaid on η(γ̇)")
ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

# compare only over the strain-rate range the flow sweep actually covers
m = (g0w >= xa.min()) & (g0w <= xa.max())
eta_i = np.exp(np.interp(np.log(g0w[m]), np.log(xa), np.log(etaa)))
logrms_dr = 100 * np.sqrt(np.mean((np.log(eta_i) - np.log(eta_p[m])) ** 2))
print("Delaware–Rutgers agreement: %.0f%% log-RMS over γ₀ω in the flow-sweep range" % logrms_dr)


Agreement within **≈9 % (log-RMS)**. The slight shortfall at the highest strain rates is
expected: beyond γ₀ ≈ 10–25 % the material leaves the linear regime and η′ begins its own
nonlinear descent.

## ✅ The LVE check: is the frequency sweep linear?

That the comparison is legitimate at all needs one check — the frequency sweep must sit
*inside* the linear viscoelastic regime. The amplitude sweep confirms it:


In [ ]:
strain = amp["strain_pct"].to_numpy()
fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(strain, amp["Gp_Pa"].to_numpy(), "bo", ms=4, label="G′")
ax.loglog(strain, amp["Gpp_Pa"].to_numpy(), "rs", ms=4, label="G″")
ax.axvline(0.5, color="k", ls="--", lw=1, label="frequency-sweep γ = 0.5 %")
ax.set_xlabel("oscillation strain / %")
ax.set_ylabel("modulus / Pa")
ax.set_title("Amplitude sweep @ 1 rad/s: linear regime check")
ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()


G′ and G″ are strain-independent up to **γ₀ ≈ 10–25 %** — the linear plateau — so the
frequency sweep's γ = 0.5 % is safely linear. Above ≈25 % the moduli roll off: the network
starts yielding and η′ can no longer stand in for the steady-shear viscosity.

## 💡 Takeaways

- **One model, three experiments.** The Carreau fit (η₀ ≈ 1.99 Pa·s, λ ≈ 0.199 s,
  n ≈ 0.414) describes the flow curve; Cox–Merz (≈2.3 %) and Delaware–Rutgers (≈9 %) show
  the oscillatory experiments were measuring the same physics all along.
- **No adjustable parameters.** Both superposition rules are parameter-free predictions —
  when they hold this well, it is strong evidence the sample is a simple, well-behaved
  linear polymer solution with no thixotropy, slip, or degradation during the measurement.
- **Oscillation as a viscometer.** Where steady shear is hard — very low rates, very high
  rates, fragile samples — a frequency or amplitude sweep plus the right rule gives you the
  flow curve anyway.

*Full write-up: the [rheofit case study](https://rheofit.readthedocs.io/en/latest/walkthrough-carreau.html).*
